## Cell 1 - Parquet merge (STANDARD PATH - use this)

Decided 2026-08-10. Combines the per-endpoint parquets produced by the ogr2ogr commands into one geoparquet. Runs in ~2 minutes. See README_Public_Extract_build.md section 5.1.

Known bugs: Land Transfer missing from the file list; folder path and output filename hardcoded per month - update both before running.

The code below combines the parquets generated from the ogr commmands together into one geoparquet with a feature class titled 'case'. 

Code takes just over a couple minutes to run

In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# Folder containing the parquet files
folder = Path(r"E:\Xentity\BLM\NLSDB_Public_Extract\Extracts\Extract_02232026")  # <-- change this as needed

files = [
    folder / "Land_Use_Authorizations_case.parquet",
    folder / "Fluid_Minerals_case.parquet",
    folder / "Solid_Minerals_case.parquet",
    folder / "Mining_Claims_case.parquet",
    folder / "Land_Tenure_case.parquet",
]

# Read each parquet into a GeoDataFrame
dfs = [gpd.read_parquet(f) for f in files]

# Concatenate and ensure it's a GeoDataFrame with the original CRS
gdf = pd.concat(dfs, ignore_index=True).pipe(gpd.GeoDataFrame, geometry="geom", crs=dfs[0].crs)

# Write combined parquet
gdf.to_parquet("nlsdb_public_extract_12082025.parquet", index=False)


## Cell 2 - Direct gpkg merge (EMERGENCY FALLBACK ONLY - do not use by preference)

Use only when ogr2ogr is unavailable or broken and cannot be fixed in time; this cell is pure GeoPandas and needs no GDAL command line. Takes just under an hour and nearly exhausts 32 GB of RAM. If used, record the month as a deviation. See README_Public_Extract_build.md section 5.4.

The code below is the workaround code created to skip the process of converting the parquets into a consolidated parquet. Instead the code below
just combines the geopackages extracted from the REST API into one "nlsdb_public_extract.gpkg". Note: ArcPro doesnt like to name "case" because "case" is a SQL term and a gpkg is a SQLite table so that needs to be called "nlsdb_case".

This code will take just under an hour to run and will almost max out your machine's RAM (32GB)

It's faster to convert each gpkg extracted from the REST API into a parquet and then combine all the parquests into one nlsdb.parquet and then convert the nlsdb.parquet into a nlsdb.gpkg. Just saying

In [4]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# Folder containing the GPKG files
folder = Path(r"E:\Xentity\BLM\NLSDB_Public_Extract\Extracts\Extract_02232026")  # <-- change if needed

files = [
    folder / "Land_Use_Authorizations_case.gpkg",
    folder / "Fluid_Minerals_case.gpkg",
    folder / "Solid_Minerals_case.gpkg",
    folder / "Mining_Claims_case.gpkg",
    folder / "Land_Tenure_case.gpkg",
]

# Read each GPKG into a GeoDataFrame (default/only layer in each)
dfs = [gpd.read_file(f) for f in files]

# Use the geometry column from the first GeoDataFrame
geom_col = dfs[0].geometry.name

# Concatenate and ensure it's a GeoDataFrame with the original CRS
combined = pd.concat(dfs, ignore_index=True)
gdf = gpd.GeoDataFrame(combined, geometry=geom_col, crs=dfs[0].crs)

# Path to the existing combined GPKG
out_gpkg = folder / "nlsdb_public_extract_02232026.gpkg"

# Write into the existing GPKG as layer "nlsdb_case"
gdf.to_file(out_gpkg, layer="nlsdb_case", driver="GPKG")


## Cell 3 - Validation: gpkg vs GDB

Compares feature counts and per-field values between the merged gpkg and the converted GDB. Run this after the GDB conversion. See README_Public_Extract_build.md section 8.4. Update the four paths at the top of the cell first.

The script below compares the nlsdb.gpkg and the nlsdb.gdb to ensure they have the same attribute and feature counts as an additional sanity check

In [ ]:
import os
import arcpy
import geopandas as gpd
import pandas as pd

# --- EDIT THESE 4 VALUES ---
gpkg_path = r"E:\Xentity\BLM\NLSDB_Public_Extract\Extracts\Extract_12072025\nlsdb_public_extract_12082025.gpkg"         # path to the GeoPackage
gpkg_layer = "nlsdb_case"                    # layer name in the GPKG

gdb_path = r"E:\Xentity\BLM\NLSDB_Public_Extract\Extracts\Extract_12072025\nlsdb_public_extract_12082025.gdb"           # path to the file geodatabase
gdb_feature_class = "nlsdb_case"             # feature class name inside the GDB
# ----------------------------

# Read GPKG layer (attributes + geometry)
gdf_gpkg = gpd.read_file(gpkg_path, layer=gpkg_layer)

# Read GDB attributes (no geometry needed for comparison)
arcpy.env.workspace = gdb_path
fc_path = os.path.join(gdb_path, gdb_feature_class)

# All non-geometry, non-OID fields
fields = [f.name for f in arcpy.ListFields(fc_path)
          if f.type not in ("Geometry", "OID")]

arr = arcpy.da.TableToNumPyArray(fc_path, fields)
df_gdb = pd.DataFrame(arr)

# Feature counts
print("=== FEATURE COUNTS ===")
print(f"GPKG  ({gpkg_layer}): {len(gdf_gpkg)}")
print(f"GDB   ({gdb_feature_class}): {len(df_gdb)}")

# Field / attribute comparison
print("\n=== FIELD NAMES ===")
gpkg_fields = set(gdf_gpkg.columns) - {"geometry"}
gdb_fields = set(df_gdb.columns)

print("Only in GPKG:", sorted(gpkg_fields - gdb_fields))
print("Only in GDB :", sorted(gdb_fields - gpkg_fields))

common_fields = sorted(gpkg_fields & gdb_fields)
print("Common fields:", common_fields)

# Attribute comparison on common fields
# (assumes same row order / same features in both datasets)
gpkg_attr = gdf_gpkg[common_fields].reset_index(drop=True)
gdb_attr = df_gdb[common_fields].reset_index(drop=True)

diff_mask = gpkg_attr.ne(gdb_attr)
rows_with_diff = diff_mask.any(axis=1).sum()
cells_with_diff = diff_mask.to_numpy().sum()

print("\n=== ATTRIBUTE DIFFERENCES (COMMON FIELDS) ===")
print(f"Rows with any differing value: {rows_with_diff}")
print(f"Total differing cells:        {cells_with_diff}")

# Optional: show a few example rows that differ
if rows_with_diff > 0:
    print("\nSample differing rows (first 5):")
    sample_mask = diff_mask.any(axis=1)
    sample_gpkg = gpkg_attr[sample_mask].head()
    sample_gdb = gdb_attr[sample_mask].head()

    display(pd.concat(
        {"GPKG": sample_gpkg, "GDB": sample_gdb},
        axis=1
    ))
